# 분류와 의사결정 오류

## Bike Sharing 고수요 시간대 분류

학습자료에서 고수요 기준을 정하고 Logistic Regression과 Decision Tree의 분류점수와 confusion matrix를 비교합니다.


## 이번 실습에서 완성할 것

- 분류점수와 임계값 0.5의 역할을 구분합니다.
- 두 모형의 지표를 같은 평가행에서 비교하고, 사전 지정한 Logistic Regression의 혼동행렬을 해석합니다.
- FP와 FN을 Bike 운영상황으로 번역합니다.

**완료 결과:** 사건비율 표, 두 모형 비교표, 기준모형 혼동행렬과 오류 해석


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI AI4I 2020 Predictive Maintenance

| 항목 | 내용 |
|---|---|
| 자료 성격 | 실제 milling 공정을 참고해 만든 합성 예지보전 자료 |
| 규모 | 10,000행, 14열 |
| 한 행의 의미 | 한 제품 처리 시점의 설비·공정 상태 |
| 입력 후보 | 제품유형, 공기·공정온도, 회전속도, torque, tool wear |
| target | `Machine failure` |
| class 분포 | 정상 9,661행, 고장 339행(3.39%) |

`UDI`와 `Product ID`는 식별자입니다. `TWF`, `HDF`, `PWF`, `OSF`, `RNF`는 `Machine failure`를 구성하는 failure-mode 결과 label이므로 feature에 넣지 않습니다. 합성자료에서 얻은 성능과 비용을 실제 설비에 그대로 일반화할 수 없습니다.

### 적용 실습: Bike 고수요 분류

시간별 `cnt`를 train 자료에서 정한 기준으로 고수요/일반수요 event로 바꿉니다. cutoff를 전체 자료에서 계산하면 test 분포를 미리 보는 누수가 생길 수 있습니다. 이번 실습은 같은 시간대의 달력·날씨 조건으로 같은 시간대의 고수요 상태를 구분하므로, FN은 실제 고수요를 일반수요로 분류한 행이고 FP는 실제 일반수요를 고수요로 분류한 행입니다. 사전 재배치 경보로 확장하려면 예측 기준시점·선행시간과 그때 이용 가능한 입력정보를 다시 정의해야 합니다.


## 실습에서 고정할 연구조건

| 그대로 유지할 방법 | Bike에서 새로 정의할 조건 |
|---|---|
| 두 모형의 같은 입력·평가행 | 동일 시간대 상태 분류라는 시간 범위 |
| `class_weight="balanced"` | FP·FN의 Bike 운영 의미 |
| 사전 지정 Logistic 상세 혼동행렬 | train의 85% 분위수 사건과 결과정보 열 제외 |

test 결과를 본 뒤 상세 해석 모형이나 사건기준을 바꾸지 않습니다.


## 실제 실행 순서와 산출물

1. 동일 시간대 상태 분류라는 질문과 입력 시점을 먼저 확인합니다.
2. train에서만 고수요 cutoff를 계산하고 train·test 사건비율을 출력합니다.
3. Logistic Regression과 Decision Tree를 같은 조건에 적합합니다.
4. 사전 지정 Logistic 혼동행렬을 읽고 TP·FP·FN·TN과 제외열을 기록합니다.

**산출물:** 사건비율 표, 두 모형 지표표, Logistic 혼동행렬과 오류 해석


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 양성 사건과 평가표본의 정의

연속값 회귀와 달리 분류는 event label과 0–1 분류점수를 만듭니다. event 정의와 cutoff가 train에서 정해졌는지 확인합니다.

### 이번 Bike 실습이 답하는 시간 질문

이번 실습은 **동일한 시간대에 관측된 달력·날씨 조건으로 그 시간대가 고수요 상태인지 구분하는 동시점 상태 분류**입니다. 뒤 시점의 수요를 미리 예측하는 실험은 아닙니다. 따라서 현재 시간의 실측 날씨를 쓰는 것은 이 질문 안에서는 미래정보 누수가 아니지만, 운영 전에 고수요를 예보하려면 예측 기준시점과 예측간격을 따로 정하고 당시 이용 가능한 과거값 또는 날씨예보로 입력을 다시 구성해야 합니다.

시간순 test는 뒤 기간에서 상태 분류 관계가 유지되는지를 확인합니다. `cnt`, `casual`, `registered`처럼 고수요 정답을 직접 만들거나 포함하는 열과 행 식별자는 입력에서 제외합니다.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

data = pd.read_csv(find_course_data("bike/hour.csv"), parse_dates=["dteday"])
data["timestamp"] = data["dteday"] + pd.to_timedelta(data["hr"], unit="h")
data = data.sort_values("timestamp").reset_index(drop=True)
cut = int(len(data) * 0.80)
train, test = data.iloc[:cut].copy(), data.iloc[cut:].copy()
HIGH_DEMAND_QUANTILE = 0.85  # 10교시에서도 같은 사건 정의를 사용합니다.
high_demand_cutoff = float(train["cnt"].quantile(HIGH_DEMAND_QUANTILE))
train["high_demand"] = train["cnt"].gt(high_demand_cutoff).astype(int)
test["high_demand"] = test["cnt"].gt(high_demand_cutoff).astype(int)
target_name = "high_demand"
numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
feature_columns = numeric_features + categorical_features
analysis_name = "Bike high-demand classification"
TASK_SCOPE = "same_hour_state_classification"
TASK_SCOPE_DESCRIPTION = "같은 시간대의 관측정보로 그 시간대의 고수요 상태를 구분"
IDENTIFIER_COLUMNS = ["instant", "dteday", "timestamp"]
DIRECT_RESULT_COLUMNS = ["cnt", "casual", "registered", "high_demand"]
input_timing = pd.DataFrame({
    "입력 묶음": ["달력정보", "날씨정보", "제외한 결과정보"],
    "열": ["season·mnth·hr·holiday·weekday·workingday", "temp·atemp·hum·windspeed·weathersit", "cnt·casual·registered"],
    "값의 시점": ["분류 대상과 같은 시간", "분류 대상과 같은 시간의 실측", "같은 시간의 수요 결과"],
    "이번 질문에서의 역할": ["입력", "입력", "정답정보이므로 제외"],
})
print("분석 시간범위:", TASK_SCOPE_DESCRIPTION)
display(input_timing)
print("이 결과는 미래 수요예보가 아니라 동일 시간대 상태 분류입니다.")
X_train, y_train = train[feature_columns], train[target_name]
X_test, y_test = test[feature_columns], test[target_name]
print("train-derived cutoff:", high_demand_cutoff)
print("train/test prevalence:", y_train.mean(), y_test.mean())
event_summary = pd.DataFrame({
    "구분": ["train", "test"],
    "전체 행": [len(train), len(test)],
    "고수요 행": [int(y_train.sum()), int(y_test.sum())],
    "고수요 비율": [y_train.mean(), y_test.mean()],
})
print("\n[연속형 cnt를 고수요 사건으로 바꾼 결과]")
print(f"기준값 {high_demand_cutoff:.1f}은 train의 85% 분위수에서만 계산했습니다.")
display(event_summary.round(3))
print("cnt, casual, registered는 사건 정의 이후 feature로 사용하지 않습니다.")


## 2. 두 분류모형을 같은 조건에서 비교하기

Logistic Regression은 입력값의 가중합을 0–1 점수로 바꾸고, Decision Tree는 입력공간을 여러 조건으로 나누어 말단구간의 점수를 만듭니다.

`class_weight="balanced"`는 학습자료가 예를 들어 정상 97건, 사건 3건처럼 불균형할 때 희귀한 사건 1건의 오분류를 더 크게 반영합니다. 사건 행을 새로 만들거나 test의 사건비율을 바꾸는 기능은 아닙니다. 이 가중치를 사용한 0–1 출력은 실제 발생확률로 바로 해석하지 않습니다.

두 모형은 같은 입력변수와 평가행에서 지표를 비교합니다. 상세 TP·FP·FN·TN 표는 test 점수를 보기 전에 정한 Logistic Regression 하나를 기준으로 해석합니다.


In [ ]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
models = {
    "logistic": Pipeline([("prep", clone(preprocess)), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))]),
    "decision tree": Pipeline([("prep", clone(preprocess)), ("model", DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=42))]),
}
for model in models.values():
    model.fit(X_train, y_train)
threshold_example = pd.DataFrame({"고수요 분류점수": [0.12, 0.48, 0.51, 0.86]})
threshold_example["threshold 0.5 예측"] = (threshold_example["고수요 분류점수"] >= 0.5).astype(int)
print("\n[분류점수 네 개를 0.5에서 class로 바꾸기]")
display(threshold_example)
print("Logistic Regression은 가중합을 sigmoid에 통과시키고, Tree는 조건분할 뒤 가중 leaf 점수를 사용합니다.")


## 3. 예측점수에서 혼동행렬까지

기본 threshold 0.5에서 각 모델의 predicted class와 false positive/negative를 계산합니다.


In [ ]:
REFERENCE_MODEL = "logistic"  # test 결과를 보기 전에 시연용으로 고정
rows, confusion_tables, classification_scores = [], {}, {}
for name, model in models.items():
    classification_score = model.predict_proba(X_test)[:, 1]
    prediction = (classification_score >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    rows.append({
        "model": name, "accuracy": accuracy_score(y_test, prediction),
        "precision": precision_score(y_test, prediction, zero_division=0),
        "recall": recall_score(y_test, prediction, zero_division=0),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    })
    confusion_tables[name] = pd.DataFrame([[tn, fp], [fn, tp]], index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"])
    classification_scores[name] = classification_score
result_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(result_table)
print("상세 오류표는 평가점수를 보기 전에 정한 Logistic Regression을 기준으로 읽습니다.")
classification_score = classification_scores[REFERENCE_MODEL]
display(confusion_tables[REFERENCE_MODEL])
score_view = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "classification_score": classification_score,
    "predicted_at_0.5": (classification_score >= 0.5).astype(int),
}, index=y_test.index)
score_view["distance_from_0.5"] = (score_view["classification_score"] - 0.5).abs()
representative_rows = pd.concat([
    score_view.nsmallest(4, "distance_from_0.5"),
    score_view.nlargest(4, "classification_score"),
]).loc[lambda frame: ~frame.index.duplicated()]
print("\n[threshold 0.5 근처와 분류점수가 높은 실제 행]")
display(representative_rows.round(3))

reference_result = result_table.loc[result_table["model"].eq(REFERENCE_MODEL)].iloc[0]
print("\n[confusion matrix를 실제 판단으로 읽기]")
print(f"상세 오류표 기준모형: {REFERENCE_MODEL}")
print(f"- TP {int(reference_result.TP)}: 실제 고수요를 고수요라고 알림")
print(f"- FN {int(reference_result.FN)}: 실제 고수요를 놓침")
print(f"- FP {int(reference_result.FP)}: 보통 수요인데 고수요라고 잘못 알림")
print(f"- TN {int(reference_result.TN)}: 보통 수요를 보통이라고 판단")


## 4. 분류분석의 유효성 점검

분류점수 범위, 두 class, 예측 길이, 결과-derived 열 제외를 확인합니다.


In [ ]:
checks = pd.Series({
    "task scope declared": TASK_SCOPE in {"same_hour_state_classification", "same_record_machine_state_classification"},
    "identifiers excluded": not (set(IDENTIFIER_COLUMNS) & set(feature_columns)),
    "direct result columns excluded": not (set(DIRECT_RESULT_COLUMNS) & set(feature_columns)),
    "binary train target": set(y_train.unique()) == {0, 1},
    "binary test target": set(y_test.unique()) == {0, 1},
    "classification score range": ((classification_score >= 0) & (classification_score <= 1)).all(),
    "same prediction length": len(classification_score) == len(y_test),
    "two comparable models": len(result_table) == 2,
    "finite metrics": np.isfinite(result_table[["accuracy", "precision", "recall"]]).all().all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("분류 실험조건 검증: PASS")

classification_review_prompt = f"""
아래에 제공한 분류실험 정보만 사용해 검토해 주세요.

- 시간 범위: {TASK_SCOPE_DESCRIPTION}
- 입력 시점 표:\n{input_timing.to_string(index=False)}
- 원자료 전체 열: {list(data.columns)}
- 결과변수: {target_name}
- 실제 입력변수: {feature_columns}
- 식별자 또는 직접 결과열: ['instant', 'dteday', 'cnt', 'casual', 'registered']
- train에서 계산한 사건 기준: cnt > {high_demand_cutoff:.1f}
- 두 모형 지표:
{result_table.to_string(index=False)}
- 사전 지정 상세 혼동행렬:
{confusion_tables[REFERENCE_MODEL].to_string()}

1. 동시점 상태 분류와 미래예측을 구분했는지, 누수 판정: 통과 / 수정 필요 / 정보 부족
2. TP·FP·FN·TN의 Bike 운영 의미
3. 균형 가중 0–1 점수를 실제 확률로 오해한 표현 점검
4. Notebook에서 직접 다시 확인할 열 또는 수치

제공되지 않은 정보는 추정하지 마세요.
""".strip()
print("\n[ChatGPT에 복사할 프롬프트]\n")
print(classification_review_prompt)


## 연구 기록

1. **이번 과제가 미래예측이 아니라 동일 시간대 상태 분류인 이유:**
2. **train에서 정한 고수요 사건기준과 train·test 사건비율:**
3. **두 모형을 비교한 동일 조건:**
4. **사전 지정 Logistic Regression의 TP·FP·FN·TN과 운영 의미:**
5. **결과변수·결과 파생열·식별자를 입력에서 제외한 근거:**
6. **가중 0–1 점수를 실제 확률로 해석하지 않은 이유:**

### 선택 검토: ChatGPT에 물어볼 때

```text
과제의 시간 범위, 입력 시점 표, 전체 열, 결과변수, 실제 입력변수, 제외열, 사건비율, 두 모형 지표와 사전 지정 Logistic Regression의 혼동행렬을 아래에 제공합니다. 동일 시간대 상태 분류와 미래예측을 구분했는지, 누수 여부와 TP·FP·FN·TN의 운영 의미를 검토해 주세요. 알 수 없는 항목은 ‘정보 부족’으로 표시하고 0–1 점수를 실제 확률로 단정하지 마세요.
```


## 완료 확인

마지막 출력이 `분류 실험조건 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 양성 사건과 cutoff는 어느 자료에서 정의했고, 결과에서 파생된 누수 열은 제외했습니까?
- 사전에 정한 기준모형의 TP·FP·FN·TN은 Bike 운영에서 각각 어떤 상황입니까?
- 균형 가중치 모형의 0–1 출력을 실제 발생확률로 단정하지 않았습니까?
